# 11g — B4.M.5: Heterogeneidade por intensidade de cana (Configuração D §6.5)

**Pré-registro v2.5 §6.5 — revisão metodológica para v2.6**

**Mudança de desenho (21/05/2026):** a H6 original do v2.3.7/v2.3.8 comparava cana-dominante (>P50 universo CS) vs cana-minoritária (<P25 universo CS). Aplicada aos 842 canavieiros, sobram apenas **1 muni minoritário** entre 194 tratados (definição universo CS leva 505 munis sem cana ao grupo minoritário). Teste binário inviável.

**Alteração:** usar **tercis intra-canavieiros** do `share_cana_eq52_pre` (P33, P66 calculados dos 842). Distribui n equilibrado e permite gradiente dose-response — argumentação causal mais forte que comparação binária. Será documentado em bloco K-extra da v2.6.

**Análise:**
- **5 outcomes:** log_solos_manejados (H6) + 4 primários B4.M.4 v2.4 (cana_direto, fert_n, calagem, res_outros)
- **1 spec:** FULL2 (consistente com 11d/11f)
- **3 tercis:** rodados independentemente com CS-DR, cada um vs todos os nunca-tratados

**Total ATTs:** 5 outcomes × 3 tercis = **15 ATTs**. Tempo esperado: ~10 min.

**Critério §6.5 Configuração D refinado:** dose-response monotônico T1 < T2 < T3 para cana_direto, com T3 sig 5%. Se confirmado, fortalece causalidade (canavieiros mais "puros" recebem mais incentivo proporcional via CBIO).

**Pré-condições no Drive:**
- `pipeline/b4m5_heterogeneidade_share.py` (este módulo)
- `pipeline/b4m4_decomposicao.py` v2.4 (join sub-canais)
- `data/interim/panel_canavieiro_main.csv`, `seeg_subcanais_panel.csv`
- `outputs_pre/share_cana_eq52_pre2018.csv` (gerado em B4.M.3)
- `data/raw/psm_baseline/base_psm_integrada_raw.csv`

## Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

Mounted at /content/drive


In [2]:
!pip install differences

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 6.4 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from differences import ATTgt
from pipeline.config import interim, out_pre
from pipeline import b4m4_decomposicao as b4m4
from pipeline import b4m5_heterogeneidade_share as b5
import importlib
importlib.reload(b4m4); importlib.reload(b5)

N_BOOT = 999
RANDOM_STATE = 42
print(f"setup OK | N_BOOT={N_BOOT}")
print(f"Outcomes B4.M.5: {b5.OUTCOMES_B4M5}")

setup OK | N_BOOT=999
Outcomes B4.M.5: ['log_solos_manejados', 'asinh_cana_direto', 'log1p_fert_n', 'log1p_calagem', 'log1p_res_outros']


## Bloco 1 — Carregar painel canônico + join sub-canais SEEG (B4.M.4)

In [4]:
panel = pd.read_csv(interim("panel_canavieiro_main.csv"), dtype={"geocode": str})
COVS_COLIDENTES = ["gini","densidade_pop","log_pop","idhm_renda",
                   "ivs_capital_humano","ivs_renda_trabalho"]
panel = panel.drop(columns=[c for c in COVS_COLIDENTES if c in panel.columns])
assert panel["geocode"].nunique() == 842
print(f"painel base: {panel.shape}")

# Join sub-canais SEEG (cana_direto)
seeg_sub = pd.read_csv(interim("seeg_subcanais_panel.csv"))
panel = b4m4.join_subchannels(panel, seeg_sub)
print(f"apos join SEEG sub-canais: {panel.shape}")

# Validar que outcomes B4.M.5 estão presentes
for o in b5.OUTCOMES_B4M5:
    if o not in panel.columns:
        print(f"  ATENCAO: {o} ausente — pode falhar abaixo")
    else:
        nn = panel[o].notna().sum()
        print(f"  OK {o:25s} notna={nn}")

painel base: (8420, 133)
  join: 842 munis no painel canônico, 842 com cana_direto computado
apos join SEEG sub-canais: (8420, 149)
  OK log_solos_manejados       notna=8410
  OK asinh_cana_direto         notna=8420
  OK log1p_fert_n              notna=8420
  OK log1p_calagem             notna=8420
  OK log1p_res_outros          notna=8420


## Bloco 2 — Construir tercis intra-canavieiros do share_cana_eq52_pre

Tercis calculados sobre os 842 canavieiros (não sobre universo CS). P33 e P66 derivados empiricamente, registrados no log.

In [6]:
panel, tercil_info = b5.build_tercil_subsets(
    panel, share_csv_path=str(out_pre("share_cana_eq52_pre2018.csv")),
)

# Salvar info dos tercis para a v2.6
pd.DataFrame([tercil_info]).to_csv(interim("b4m5_tercil_info.csv"), index=False)
print(f"\nOK tercis adicionados: {panel.shape}")
print(f"  P33 = {tercil_info['p33']:.4f}")
print(f"  P66 = {tercil_info['p66']:.4f}")

  P33 (intra-canavieiros) = 0.9643
  P66 (intra-canavieiros) = 0.9962

  Distribuição dos 842 canavieiros por tercil:
    T1_baixo: 278 munis
    T2_medio: 277 munis
    T3_alto: 286 munis
    sem tercil (NaN): 1 munis

  Tratados (is_treated_ever=True) por tercil:
    T1_baixo: 36 tratados
    T2_medio: 62 tratados
    T3_alto: 96 tratados

OK tercis adicionados: (8420, 151)
  P33 = 0.9643
  P66 = 0.9962


## Bloco 3 — Reconstruir covariáveis e painel CS (idêntico ao 11d/11f)

In [7]:
psm_raw = pd.read_csv(
    BASE_DIR / "data/raw/psm_baseline/base_psm_integrada_raw.csv",
    low_memory=False,
)
psm_raw["geocode"] = psm_raw["0_cd_ibge"].astype(str).str.zfill(7)

muni_id = (panel.groupby("geocode", as_index=False)
           .agg(municipio=("municipio","first"), uf=("uf","first"),
                is_treated_ever=("is_treated_ever","first"),
                g_m=("g_m","first"), bioma=("bioma","first")))
muni_id["treated"] = muni_id["is_treated_ever"].astype(int)
df_cs = muni_id.merge(psm_raw, on="geocode", how="inner")
print(f"df_cs: {df_cs.shape}")

df_cs: (842, 172)


In [8]:
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors="coerce") if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors="coerce") if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)

def build_covariates_raw(df):
    d = df.copy(); idx = d.index
    d["log_pib_total"]=safe_log1p(d.get("1_pib_total"),idx)
    d["log_pib_pc"]=safe_log1p(d.get("1_pib_percap"),idx)
    d["log_pop"]=safe_log1p(d.get("2_pop_2017_ibge"),idx)
    d["log_area_total"]=safe_log1p(d.get("14_area_total"),idx)
    d["densidade_pop"]=safe_div(d.get("2_pop_2017_ibge"),d.get("14_area_total"),idx)
    d["share_vadc_agro"]=safe_div(d.get("1_vadc_agro"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_ind"]=safe_div(d.get("1_vadc_ind"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_serv"]=safe_div(d.get("1_vadc_serv"),d.get("1_vadc_bruto"),idx)
    d["share_vadc_adm"]=safe_div(d.get("1_vadc_adm"),d.get("1_vadc_bruto"),idx)
    d["share_cana_baseline"]=asn(d.get("3_mb_sharegrp_pre_cana"),idx)
    d["mb_share_soja"]=asn(d.get("3_mb_sharegrp_pre_soja"),idx)
    d["mb_share_pastagem"]=asn(d.get("3_mb_sharegrp_pre_pastagem"),idx)
    d["mb_share_vegetacao_nativa"]=asn(d.get("3_mb_sharegrp_pre_vegetacao_nativa"),idx)
    d["mb_share_urbano"]=asn(d.get("3_mb_sharegrp_pre_urbano_infra"),idx)
    d["log_area_cana"]=safe_log1p(d.get("4_area_colhida_ha_cana"),idx)
    d["log_area_soja"]=safe_log1p(d.get("4_area_colhida_ha_soja"),idx)
    d["log_area_agri_total"]=safe_log1p(d.get("4_area_colhida_ha"),idx)
    d["share_area_cana_agri"]=safe_div(d.get("4_area_colhida_ha_cana"),d.get("4_area_colhida_ha"),idx)
    d["share_est_af"]=safe_div(d.get("5_num_est_af"),d.get("5_num_est_total"),idx)
    d["share_est_mp"]=safe_div(d.get("5_num_est_mp"),d.get("5_num_est_total"),idx)
    d["share_area_af"]=safe_div(d.get("6_area_lav_af"),d.get("6_area_lav_total"),idx)
    d["share_area_mp"]=safe_div(d.get("6_area_lav_mp"),d.get("6_area_lav_total"),idx)
    d["trator_per_est"]=safe_div(d.get("11_num_trator_total"),d.get("5_num_est_total"),idx)
    d["share_est_irrig"]=safe_div(d.get("12_num_est_irrig_total"),d.get("5_num_est_total"),idx)
    d["share_area_irrig"]=safe_div(d.get("12_area_irrig_total"),d.get("6_area_lav_total"),idx)
    d["share_est_fin_total"]=safe_div(d.get("13_num_est_fin_total"),d.get("5_num_est_total"),idx)
    d["share_est_at"]=safe_div(d.get("10_num_est_receb_at"),d.get("5_num_est_total"),idx)
    d["natveg_share_area"]=safe_div(d.get("14_vegetacao_natural"),d.get("14_area_total"),idx)
    d["desmat_share_area"]=safe_div(d.get("14_desmatado"),d.get("14_area_total"),idx)
    d["idhm_renda"]=asn(d.get("17_idhm_renda"),idx); d["idhm_educ"]=asn(d.get("17_idhm_educ"),idx)
    d["ivs_infra"]=asn(d.get("17_ivs_infraestrutura_urbana"),idx); d["gini"]=asn(d.get("17_i_gini"),idx)
    d["idhm_long"]=asn(d.get("17_idhm_long"),idx); d["ivs_capital_humano"]=asn(d.get("17_ivs_capital_humano"),idx)
    d["ivs_renda_trabalho"]=asn(d.get("17_ivs_renda_e_trabalho"),idx)
    d["share_est_at_coop"]=safe_div(d.get("10_num_est_receb_at_coop"),d.get("5_num_est_total"),idx)
    d["share_est_at_gov"]=safe_div(d.get("10_num_est_receb_at_gov"),d.get("5_num_est_total"),idx)
    d["share_fin_invest"]=safe_div(d.get("13_num_est_fin_invest"),d.get("5_num_est_total"),idx)
    d["share_fin_cust"]=safe_div(d.get("13_num_est_fin_cust"),d.get("5_num_est_total"),idx)
    d["share_est_trator"]=safe_div(d.get("11_num_est_trator_total"),d.get("5_num_est_total"),idx)
    d["share_est_irrig_pivo"]=safe_div(d.get("12_num_est_irrig_pivo"),d.get("5_num_est_total"),idx)
    d["pct_est_energia"]=asn(d.get("7_est_com_energia%"),idx)
    return d

df_cs = build_covariates_raw(df_cs)

COVS_LEAN = ["log_pib_total","log_pib_pc","log_pop","densidade_pop",
             "share_vadc_agro","share_vadc_ind","share_vadc_serv",
             "share_cana_baseline","mb_share_soja","mb_share_pastagem","mb_share_vegetacao_nativa",
             "log_area_cana","log_area_soja","share_area_cana_agri",
             "share_est_af","share_area_af","trator_per_est","share_est_irrig","share_est_fin_total",
             "ivs_infra","gini"]
COVS_FULL = COVS_LEAN + ["share_vadc_adm","idhm_educ","idhm_renda","idhm_long",
                          "ivs_capital_humano","ivs_renda_trabalho",
                          "share_est_at","share_est_at_coop","share_est_at_gov",
                          "share_fin_invest","share_fin_cust",
                          "share_est_trator","share_est_irrig_pivo","pct_est_energia"]
COVS_FULL2 = [c for c in COVS_FULL if c not in ("share_vadc_agro","share_vadc_ind")]

# Imputação por mediana estadual
for c in COVS_FULL2:
    if df_cs[c].isna().any():
        df_cs[c] = df_cs.groupby("uf")[c].transform(lambda x: x.fillna(x.median()))
        df_cs[c] = df_cs[c].fillna(df_cs[c].median())
assert df_cs[COVS_FULL2].isna().sum().sum() == 0
print(f"OK FULL2: {len(COVS_FULL2)} covs imputadas")

# Painel CS com bug 1: NaN = never-treated
panel_cs = panel.merge(df_cs[["geocode"] + COVS_FULL2], on="geocode", how="left")
panel_cs["g_m_cs"] = panel_cs["g_m"]
n_treated = panel_cs["g_m_cs"].notna().sum() // 10
n_never = panel_cs["g_m_cs"].isna().sum() // 10
assert n_treated == 194 and n_never == 648
print(f"OK painel CS: {n_treated} tratados, {n_never} nunca-tratados")

OK FULL2: 33 covs imputadas
OK painel CS: 194 tratados, 648 nunca-tratados


## Bloco 4 — CS-DR por tercil × 5 outcomes (15 ATTs)

Cada tercil rodado independentemente. Amostra de cada ATT: tratados do tercil + todos nunca-tratados.

In [9]:
cs_b4m5 = b5.run_csdr_by_tercil(
    panel_cs,
    outcomes=b5.OUTCOMES_B4M5,
    covs=COVS_FULL2,
    n_boot=N_BOOT,
    random_state=RANDOM_STATE,
)
cs_b4m5.to_csv(interim("att_b4m5_por_tercil.csv"), index=False)
print(f"\nOK att_b4m5_por_tercil.csv {cs_b4m5.shape}")
cs_b4m5


>>> log_solos_manejados
  T1_baixo   ATT = +0.0322  SE=0.0320  |z|=1.01   (n_trat=36)  [4.8s]
  T2_medio   ATT = +0.0785  SE=0.0341  |z|=2.30*  (n_trat=62)  [2.5s]
  T3_alto    ATT = +0.0452  SE=0.0306  |z|=1.47   (n_trat=96)  [2.1s]

>>> asinh_cana_direto
  T1_baixo   ATT = +0.2767  SE=0.2458  |z|=1.13   (n_trat=36)  [1.8s]
  T2_medio   ATT = +0.1671  SE=0.0808  |z|=2.07*  (n_trat=62)  [2.5s]
  T3_alto    ATT = +0.0458  SE=0.0691  |z|=0.66   (n_trat=96)  [6.6s]

>>> log1p_fert_n
  T1_baixo   ATT = +0.1726  SE=0.1210  |z|=1.43   (n_trat=36)  [1.8s]
  T2_medio   ATT = +0.1401  SE=0.0600  |z|=2.34*  (n_trat=62)  [2.5s]
  T3_alto    ATT = +0.0428  SE=0.0483  |z|=0.89   (n_trat=96)  [2.1s]

>>> log1p_calagem
  T1_baixo   ATT = -0.0206  SE=0.0359  |z|=0.57   (n_trat=36)  [1.7s]
  T2_medio   ATT = +0.0570  SE=0.0288  |z|=1.98*  (n_trat=62)  [7.2s]
  T3_alto    ATT = +0.0310  SE=0.0300  |z|=1.04   (n_trat=96)  [2.1s]

>>> log1p_res_outros
  T1_baixo   ATT = -0.0558  SE=0.0569  |z|=0.98   (n_

,outcome,tercil,ATT,SE,z,sig_5pct,CI_lo,CI_hi,n_munis_total,n_tratados_tercil
0,log_solos_manejados,T1_baixo,0.032178,0.031981,1.006166,False,-0.030503,0.094860,683,36
1,log_solos_manejados,T2_medio,0.078510,0.034096,2.302612,True,0.011683,0.145337,709,62
2,log_solos_manejados,T3_alto,0.045167,0.030623,1.474910,False,-0.014854,0.105188,743,96
3,asinh_cana_direto,T1_baixo,0.276715,0.245821,1.125678,False,-0.205085,0.758516,684,36
4,asinh_cana_direto,T2_medio,0.167146,0.080781,2.069137,True,0.008819,0.325474,710,62
5,asinh_cana_direto,T3_alto,0.045773,0.069113,0.662286,False,-0.089687,0.181232,744,96
6,log1p_fert_n,T1_baixo,0.172632,0.121045,1.426182,False,-0.064611,0.409875,684,36
7,log1p_fert_n,T2_medio,0.140146,0.059999,2.335784,True,0.022549,0.257742,710,62
8,log1p_fert_n,T3_alto,0.042799,0.048278,0.886514,False,-0.051824,0.137422,744,96
9,log1p_calagem,T1_baixo,-0.020593,0.035898,0.573643,False,-0.090952,0.049767,684,36


## Bloco 5 — Diagnóstico de dose-response

In [10]:
dr = b5.assess_dose_response(cs_b4m5)
dr.to_csv(interim("att_b4m5_dose_response.csv"), index=False)

print("="*108)
print("DIAGNOSTICO DE DOSE-RESPONSE — gradiente T1 < T2 < T3 por outcome")
print("="*108)
print(f"{'outcome':<22}{'T1':>9}{'sig':>5}{'T2':>9}{'sig':>5}{'T3':>9}{'sig':>5}"
      f"{'T3-T1':>9}{'mono':>6}  veredito")
print("-"*108)
for _, r in dr.iterrows():
    def fmt(att, sig):
        if pd.isna(att): return "    nan   "
        return f"{att:+.4f}{'*' if sig else ' '}"
    print(f"{r['outcome']:<22}"
          f"{r['ATT_T1']:+9.4f}{'*' if r['sig_T1'] else ' ':>5}"
          f"{r['ATT_T2']:+9.4f}{'*' if r['sig_T2'] else ' ':>5}"
          f"{r['ATT_T3']:+9.4f}{'*' if r['sig_T3'] else ' ':>5}"
          f"{r['delta_T3_T1']:+9.4f}{r['monotonia']:>6}  {r['veredito']}")
print()
print("(*) sig 5% bicaudal")
print()
print("LEGENDA dos vereditos:")
print("  DOSE_RESPONSE_POSITIVO  : T1<T2<T3 monotonico e T3 sig 5%")
print("  DOSE_RESPONSE_NEGATIVO  : T1>T2>T3 monotonico e T3 sig 5%")
print("  SEM_DOSE_RESPONSE_SIG   : T3 sig mas sem gradiente claro")
print("  PARCIAL                 : gradiente sem T3 sig")
print("  SEM_EFEITO              : nenhum tercil sig")

DIAGNOSTICO DE DOSE-RESPONSE — gradiente T1 < T2 < T3 por outcome
outcome                      T1  sig       T2  sig       T3  sig    T3-T1  mono  veredito
------------------------------------------------------------------------------------------------------------
log_solos_manejados     +0.0322       +0.0785    *  +0.0452       +0.0130     0  MISTO
asinh_cana_direto       +0.2767       +0.1671    *  +0.0458       -0.2309     -  PARCIAL
log1p_fert_n            +0.1726       +0.1401    *  +0.0428       -0.1298     -  PARCIAL
log1p_calagem           -0.0206       +0.0570    *  +0.0310       +0.0516     0  MISTO
log1p_res_outros        -0.0558       -0.0328       -0.0526       +0.0032     0  SEM_EFEITO

(*) sig 5% bicaudal

LEGENDA dos vereditos:
  DOSE_RESPONSE_POSITIVO  : T1<T2<T3 monotonico e T3 sig 5%
  DOSE_RESPONSE_NEGATIVO  : T1>T2>T3 monotonico e T3 sig 5%
  SEM_DOSE_RESPONSE_SIG   : T3 sig mas sem gradiente claro
  PARCIAL                 : gradiente sem T3 sig
  SEM_EFEITO      

## Bloco 6 — Síntese da Configuração D §6.5

In [11]:
print("="*78)
print("CONFIGURACAO D §6.5 — sintese editorial")
print("="*78)

# Outcome chave: cana_direto (Configuração I do v2.4)
cd = dr[dr["outcome"] == "asinh_cana_direto"].iloc[0]
print(f"\n*** OUTCOME CHAVE: asinh_cana_direto ***")
print(f"  ATT_T1 = {cd['ATT_T1']:+.4f} (sig: {bool(cd['sig_T1'])})")
print(f"  ATT_T2 = {cd['ATT_T2']:+.4f} (sig: {bool(cd['sig_T2'])})")
print(f"  ATT_T3 = {cd['ATT_T3']:+.4f} (sig: {bool(cd['sig_T3'])})")
print(f"  Delta (T3-T1) = {cd['delta_T3_T1']:+.4f}")
print(f"  Monotonia = {cd['monotonia']}")
print(f"  Veredito = {cd['veredito']}")
print()
if cd["veredito"] == "DOSE_RESPONSE_POSITIVO":
    print("  >>> CONFIGURACAO D CONFIRMADA <<<")
    print("  Efeito do RenovaBio em cana_direto aumenta com intensidade de cana.")
    print("  Dose-response causal claro: canavieiros mais puros (T3) recebem mais")
    print("  incentivo proporcional via CBIO — assinatura empirica esperada de uma")
    print("  politica que opera via produção certificada de cana.")
elif cd["veredito"] == "PARCIAL":
    print("  >>> CONFIGURACAO D PARCIAL <<<")
    print("  Gradiente monotonico presente mas significancia em T3 ausente.")
    print("  Compativel com efeito heterogeneo, magnitude insuficiente para identificacao.")
else:
    print(f"  >>> CONFIGURACAO D NAO CONFIRMADA ({cd['veredito']}) <<<")
    print("  Sem gradiente dose-response claro. Heterogeneidade por share-cana")
    print("  nao explica a magnitude do ATT agregado — outras dimensoes podem")
    print("  ser mais relevantes (cohort, UF, baseline de mecanizacao).")

# Resumo todos os outcomes
print(f"\n--- todos os 5 outcomes ---")
for _, r in dr.iterrows():
    print(f"  {r['outcome']:<22} {r['veredito']:<26} (T3-T1 = {r['delta_T3_T1']:+.4f})")

CONFIGURACAO D §6.5 — sintese editorial

*** OUTCOME CHAVE: asinh_cana_direto ***
  ATT_T1 = +0.2767 (sig: False)
  ATT_T2 = +0.1671 (sig: True)
  ATT_T3 = +0.0458 (sig: False)
  Delta (T3-T1) = -0.2309
  Monotonia = -
  Veredito = PARCIAL

  >>> CONFIGURACAO D PARCIAL <<<
  Gradiente monotonico presente mas significancia em T3 ausente.
  Compativel com efeito heterogeneo, magnitude insuficiente para identificacao.

--- todos os 5 outcomes ---
  log_solos_manejados    MISTO                      (T3-T1 = +0.0130)
  asinh_cana_direto      PARCIAL                    (T3-T1 = -0.2309)
  log1p_fert_n           PARCIAL                    (T3-T1 = -0.1298)
  log1p_calagem          MISTO                      (T3-T1 = +0.0516)
  log1p_res_outros       SEM_EFEITO                 (T3-T1 = +0.0032)


## Conclusão B4.M.5

Saídas:
- `att_b4m5_por_tercil.csv` — 5 outcomes × 3 tercis = 15 ATTs
- `att_b4m5_dose_response.csv` — síntese com veredito por outcome
- `b4m5_tercil_info.csv` — P33, P66, n por tercil (rastreabilidade)

**Próximo passo:** se DOSE_RESPONSE_POSITIVO em cana_direto, fechamos B4.M.5 com argumento causal robusto. Em seguida:
- B4.M.7: balanço CO₂eq queima vs cana_direto (GWP IPCC AR6)
- Consolidar v2.6 do pré-registro com bloco K-extra (revisão tercis)
- Decidir sobre PAM 1613 (culturas permanentes) para próxima sessão

In [12]:
# ============================================================================
# CÉLULA ADICIONAL para 11g — event-study por tercil
# Cole APÓS o Bloco 6 do 11g. Reusa panel_cs, COVS_FULL2, N_BOOT, RANDOM_STATE.
# Tempo: ~3-5 min (3 event-studies × ~60s cada, sob FULL2 com 999 boot).
# ============================================================================
import time
import numpy as np
import pandas as pd
from differences import ATTgt
from pipeline import b4_event_studies as bes
import importlib; importlib.reload(bes)

OUTCOME_FOCO = "asinh_cana_direto"
TERCIS = ["T1_baixo", "T2_medio", "T3_alto"]

print("=" * 78)
print(f"EVENT-STUDY POR TERCIL — outcome: {OUTCOME_FOCO}")
print("=" * 78)
print(f"Spec: FULL2 | N_BOOT={N_BOOT} | janela t=[-4, +4]")
print()

# Identificar never-treated (g_m_cs = NaN, Bug 1)
never_geocodes = panel_cs[panel_cs["g_m_cs"].isna()]["geocode"].unique()

rows_event = []
rows_pretrend = []

for tercil in TERCIS:
    t0 = time.time()
    print(f">>> {tercil}")
    try:
        # Subset: tratados deste tercil + todos nunca-tratados
        tratados_tercil = (
            panel_cs[
                (~panel_cs["g_m_cs"].isna())
                & (panel_cs["tercil_share_cana"] == tercil)
            ]["geocode"].unique()
        )
        amostra = list(tratados_tercil) + list(never_geocodes)
        data = (
            panel_cs[panel_cs["geocode"].isin(amostra)]
            .dropna(subset=[OUTCOME_FOCO])
            .set_index(["geocode", "ano"])
            .sort_index()
        )
        n_munis = data.index.get_level_values("geocode").nunique()
        n_trat = len(tratados_tercil)

        attgt = ATTgt(data=data, cohort_column="g_m_cs")
        attgt.fit(
            formula=f"{OUTCOME_FOCO} ~ " + " + ".join(COVS_FULL2),
            est_method="dr",
            control_group="never_treated",
            boot_iterations=N_BOOT,
            random_state=RANDOM_STATE,
            progress_bar=False,
            n_jobs=1,
        )
        ev_raw = attgt.aggregate("event")
        ev = bes._flatten_event_result(ev_raw)
        # Aplicar janela [-4, +4] (t+5 inutil — vimos no 11f)
        ev = ev[(ev["event_time"] >= -4) & (ev["event_time"] <= 4)].copy()
        ev["outcome"] = OUTCOME_FOCO
        ev["tercil"] = tercil
        ev["spec"] = "FULL2"
        ev["n_munis_total"] = n_munis
        ev["n_tratados_tercil"] = n_trat
        rows_event.append(ev)
        print(f"  OK  ({n_trat} tratados, {n_munis} total)  [{time.time()-t0:.1f}s]")
    except Exception as e:
        print(f"  XX FALHOU: {type(e).__name__}: {str(e)[:80]}")

if not rows_event:
    raise RuntimeError("Nenhum event-study rodou.")

ev_tercis = pd.concat(rows_event, ignore_index=True)
ev_tercis.to_csv(interim("att_b4m5_event_por_tercil.csv"), index=False)

# Pre-trends por tercil
print()
print("=" * 78)
print("TESTE DE PRE-TRENDS POR TERCIL")
print("=" * 78)
pre_tercis_rows = []
for tercil in TERCIS:
    sub = ev_tercis[ev_tercis["tercil"] == tercil].copy()
    if sub.empty:
        continue
    pre = sub[(sub["event_time"] >= -4) & (sub["event_time"] <= -2)].dropna(
        subset=["ATT", "SE"])
    pre = pre[pre["SE"] > 0]
    if pre.empty:
        flag = "NO_DATA"
        wald = pval = n_sig = np.nan
    else:
        z = (pre["ATT"] / pre["SE"]).abs()
        n_sig = int((z > 1.959963985).sum())
        wald = float((z ** 2).sum())
        from scipy.stats import chi2
        pval = float(1 - chi2.cdf(wald, df=len(pre)))
        if n_sig == 0 and pval > 0.10:
            flag = "PRE_TRENDS_PLANOS"
        elif n_sig == 0 and pval > 0.05:
            flag = "PRE_TRENDS_MARGINAIS"
        else:
            flag = "PRE_TRENDS_VIOLADOS"

    pre_tercis_rows.append({
        "tercil": tercil,
        "n_pre": len(pre),
        "n_sig_individual": n_sig,
        "wald_stat": wald,
        "p_value": pval,
        "flag": flag,
    })

pre_tercis = pd.DataFrame(pre_tercis_rows)
print(f"{'tercil':<12}{'n_pre':>7}{'n_sig':>7}{'wald':>9}{'p-val':>9}  flag")
print("-" * 56)
for _, r in pre_tercis.iterrows():
    print(f"{r['tercil']:<12}{int(r['n_pre']):>7}{int(r['n_sig_individual']):>7}"
          f"{r['wald_stat']:>9.2f}{r['p_value']:>9.4f}  {r['flag']}")

# Tabela completa do event-study por tercil
print()
print("=" * 78)
print("EVENT-STUDY COMPLETO POR TERCIL (t=-4 a +4)")
print("=" * 78)
print(f"{'tercil':<12}{'t':>4}{'ATT':>10}{'SE':>10}{'|z|':>7}"
      f"{'CI_lo':>10}{'CI_hi':>10}  sig")
print("-" * 78)
for tercil in TERCIS:
    sub = ev_tercis[ev_tercis["tercil"] == tercil].sort_values("event_time")
    for _, r in sub.iterrows():
        att = float(r["ATT"]) if pd.notna(r["ATT"]) else np.nan
        se = float(r["SE"]) if pd.notna(r["SE"]) else np.nan
        z = abs(att / se) if (np.isfinite(se) and se > 0) else np.nan
        sig = "  *" if (np.isfinite(z) and z > 1.96) else ""
        ci_lo = float(r.get("CI_lo", np.nan)) if pd.notna(r.get("CI_lo")) else np.nan
        ci_hi = float(r.get("CI_hi", np.nan)) if pd.notna(r.get("CI_hi")) else np.nan
        print(f"{tercil:<12}{int(r['event_time']):>4}{att:>+10.4f}{se:>10.4f}"
              f"{z:>7.2f}{ci_lo:>+10.4f}{ci_hi:>+10.4f}{sig}")
    print()

# Síntese final
print()
print("=" * 78)
print("SINTESE: pre-trends por tercil ajudam a discriminar Leitura A vs C")
print("=" * 78)
print()
print("Leitura A (mecanizacao pre-existente em T3) preve:")
print("  - T1 (parciais): pre-trends planos, ATT pos pequeno")
print("  - T2 (medios): pre-trends planos, ATT pos significante")
print("  - T3 (puros): pre-trends planos OU positivos (ja em transicao antes)")
print("  - se T3 tem pre-trends planos: confirma A (sem efeito = sem espaco")
print("    para mecanizar)")
print("  - se T3 tem pre-trends violados positivos: confirma A reforcadamente")
print("    (a transicao ja aconteceu antes de 2018)")
print()
print("Leitura C (confounding por trajetoria em T3) preve:")
print("  - T3 com pre-trends violados positivos especificamente")
print("  - T1 e T2 com pre-trends planos")
print()
print("Compara o flag de cada tercil para decidir.")

EVENT-STUDY POR TERCIL — outcome: asinh_cana_direto
Spec: FULL2 | N_BOOT=999 | janela t=[-4, +4]

>>> T1_baixo
  OK  (36 tratados, 684 total)  [8.4s]
>>> T2_medio
  OK  (62 tratados, 710 total)  [2.6s]
>>> T3_alto
  OK  (96 tratados, 744 total)  [6.3s]

TESTE DE PRE-TRENDS POR TERCIL
tercil        n_pre  n_sig     wald    p-val  flag
--------------------------------------------------------
T1_baixo          3      0     1.16   0.7620  PRE_TRENDS_PLANOS
T2_medio          3      1     6.36   0.0954  PRE_TRENDS_VIOLADOS
T3_alto           3      0     1.99   0.5740  PRE_TRENDS_PLANOS

EVENT-STUDY COMPLETO POR TERCIL (t=-4 a +4)
tercil         t       ATT        SE    |z|     CI_lo     CI_hi  sig
------------------------------------------------------------------------------
T1_baixo      -4   +0.0927    0.1280   0.72   -0.1581   +0.3436
T1_baixo      -3   -0.1385    0.2615   0.53   -0.6511   +0.3741
T1_baixo      -2   +0.0975    0.1632   0.60   -0.2223   +0.4173
T1_baixo      -1   +0.0579  